<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: When a simple model is better

**[When a simple model is better](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/)** · Machine Learning: From Problem to Reliable Model · Module 7, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare a points score, logistic regression and gradient boosting under the same capacity, explain one ticket with each, and decide with a checklist when a simple model is the better choice.

| | |
|---|---|
| **Time** | About 60 minutes |
| **Needs** | An internet connection for the setup cells. No account. |
| **You should know** | Contributions of one ticket, from [Explain predictions](https://learning.nextia-ai.com/courses/ml/m07/explain-predictions/). Boosting and the cost of complexity, from [Ensembles](https://learning.nextia-ai.com/courses/ml/m05/ensembles/#the-cost-of-complexity). The one-standard-error rule, from [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/#the-one-standard-error-rule). |
| **You do not need** | The local project. The setup below downloads the data and the course's `ticket_model.py`. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

The lesson page has the real-world cases (credit, the EU rules, banks, hospitals, insurance) with their sources. This notebook has the Larkfield part.

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M07-L03-when-a-simple-model-is-better/when-a-simple-model-is-better-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. It is the final version from Select useful features (Module 6), with 11 features. The next cells import it.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A points score on six tickets

A bank refuses a loan, and the customer asks "why?". If the bank uses a points card, the answer is easy: "you lost points for three late payments". A **points score** is a checklist: each fact adds a fixed number of points, and a person can add them up by hand.

This lesson uses a points score for Larkfield's tickets:

- priority 1: **+2 points**;
- team payment or warranty: **+1 point**;
- at least one escalation of the same customer in the last 90 days: **+1 point**.

The next cell defines `points()` and scores six made-up tickets.

> **Predict.** Add up the points of tickets A and C by hand. Then run the cell.

In [ ]:
def points(df):
    """The points score: a checklist that a person can add up by hand."""
    return (2 * (df["priority"] == 1)
            + df["team"].isin(["payment", "warranty"]).astype(int)
            + (df["prior_escalations_90d"] >= 1).astype(int))


tiny = pd.DataFrame({"priority": [1, 1, 2, 3, 2, 3],
                     "team": ["payment", "delivery", "warranty", "returns", "payment", "account"],
                     "prior_escalations_90d": [1, 0, 2, 0, 0, 1]}, index=list("ABCDEF"))
tiny["points"] = points(tiny)
tiny

> **Check.** You should see 4 points for A (2 + 1 + 1), 2 for B, 2 for C (0 + 1 + 1), 0 for D, 1 for E and 1 for F.

Every point has a reason that you can say in words. That is the whole model.

## 2. Why these rules?

A points score is not a guess: each rule comes from the train data. The next cell loads the data and compares the escalation rate of the tickets that meet each rule with the other tickets.

> **Check.** You should see `all tickets: 0.117`, then priority 1 at 0.291 (others 0.089), payment or warranty at 0.185 (others 0.089), and an earlier escalation at 0.148 (others 0.091).

In [ ]:
from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
print("all tickets:", round(train[TARGET].mean(), 3))
rules = {"priority 1": train["priority"] == 1,
         "payment or warranty": train["team"].isin(["payment", "warranty"]),
         "an escalation in the last 90 days": train["prior_escalations_90d"] >= 1}
for name, rows in rules.items():
    print(f"{name}: {train.loc[rows, TARGET].mean():.3f}, other tickets: {train.loc[~rows, TARGET].mean():.3f}")

Priority 1 more than triples the rate, so it gets 2 points. The two other rules roughly double or add half again, so they get 1 point each.

Now score the validation month (May 2026), the data we keep to check models. The next cell adds a `points` column and shows, for each number of points, the tickets (`size`), the escalations (`sum`) and the rate (`mean`).

> **Predict.** Does the rate go up with every extra point?

In [ ]:
valid["points"] = points(valid)
valid.groupby("points")[TARGET].agg(["size", "sum", "mean"]).round(3)

> **Check.** You should see five rows, from 0 points (320 tickets, rate 0.053) to 4 points (32 tickets, rate 0.562).

The rate goes up with every point. A checklist of three lines already ranks the tickets.

## 3. Three models, one capacity

Now compare three models in the same way:

- the **points score** (no training);
- **logistic regression**, the final model of the course;
- **gradient boosting** with learning rate 0.1 and depth 2, the best setting of the search in [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/#the-one-standard-error-rule).

The next cell fits the two trained models on the train set and scores May. It takes a few seconds.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

logistic = build_pipeline().fit(train[FEATURES], train[TARGET])
boosting = build_pipeline(HistGradientBoostingClassifier(max_depth=2, learning_rate=0.1, random_state=0))
boosting.fit(train[FEATURES], train[TARGET])
scores = {"points": valid["points"],
          "logistic": logistic.predict_proba(valid[FEATURES])[:, 1],
          "boosting": boosting.predict_proba(valid[FEATURES])[:, 1]}

The senior team can review at most 20% of new tickets. The next cell flags tickets with the lowest threshold that keeps the flags at or under 20%, the same rule as in [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m06/thresholds-and-calibration/). A points score has only whole numbers, so its thresholds are 0, 1, 2, 3, 4 and 5.

> **Predict.** Which model catches the most escalations? By how many?

In [ ]:
def at_capacity(name, score, steps, capacity=0.20):
    """Flag with the lowest threshold that flags at most `capacity`, and print the result."""
    threshold = next(t for t in steps if (score >= t).mean() <= capacity)
    flags = score >= threshold
    caught = int((flags & (valid[TARGET] == 1)).sum())
    print(f"{name:9s} threshold {threshold:<5} flagged {flags.mean():.1%}  caught {caught}"
          f"  precision {caught / flags.sum():.3f}  recall {caught / valid[TARGET].sum():.3f}")


at_capacity("points", scores["points"], [0, 1, 2, 3, 4, 5])
steps = [s / 100 for s in range(1, 100)]
at_capacity("logistic", scores["logistic"], steps)
at_capacity("boosting", scores["boosting"], steps)

> **Check.** You should see `points` with threshold 3, flagged 9.8%, caught 53, recall 0.290; `logistic` with threshold 0.19, flagged 19.3%, caught 86, recall 0.470; and `boosting` with threshold 0.18, flagged 19.1%, caught 86, recall 0.470.

Logistic regression and boosting catch **the same 86 escalations** of 183. The points score flags only 9.8%: with 3 or more points, 116 tickets; with 2 or more, 327 tickets, which is 27.7% and over the capacity. A score with few values cannot hit 20% exactly. That is a real cost of a coarse checklist.

The next cell measures each model's ranking over all thresholds with average precision (AP).

In [ ]:
from sklearn.metrics import average_precision_score

for name, score in scores.items():
    print(name, round(average_precision_score(valid[TARGET], score), 3))

> **Check.** You should see `points 0.31`, `logistic 0.391` and `boosting 0.395`.

The gap between logistic regression and boosting is 0.004. In [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/#the-one-standard-error-rule), the cross-validation gap was 0.0031 with a standard error of 0.008: smaller than the noise.

To use the whole capacity with the points score, Grace's team can add a **tie rule**: review all tickets with 3 or 4 points, then the 2-point tickets in order of arrival while there is room. The next cell does that.

In [ ]:
order = valid.sort_values(["points", "created_at"], ascending=[False, True])
top = order.head(int(0.20 * len(valid)))
caught = int(top[TARGET].sum())
print(len(top), "flagged, caught", caught, "| recall", round(caught / valid[TARGET].sum(), 3))

> **Check.** You should see `235 flagged, caught 84 | recall 0.459`.

With a tie rule, a three-line checklist catches 84 escalations, two fewer than the trained models. On this data, the price of the simplest model is small.

## 4. One ticket, three explanations

Omar asks about T-117014, a priority 1 payment ticket that escalated: "Why was it flagged?" The next cell shows what each model can tell him.

In [ ]:
def point_reasons(ticket):
    """The points of one ticket, line by line."""
    row = ticket.iloc[0]
    return {"priority 1 (+2)": 2 * int(row["priority"] == 1),
            "payment or warranty (+1)": int(row["team"] in ("payment", "warranty")),
            "an escalation in the last 90 days (+1)": int(row["prior_escalations_90d"] >= 1)}


ticket = valid[valid["ticket_id"] == "T-117014"]
print(point_reasons(ticket))
print("logistic score:", round(logistic.predict_proba(ticket[FEATURES])[0, 1], 3))
print("boosting score:", round(boosting.predict_proba(ticket[FEATURES])[0, 1], 3),
      "from", boosting.named_steps["model"].n_iter_, "trees")

> **Check.** You should see 2 + 1 + 1 points (4 in all), `logistic score: 0.38` and `boosting score: 0.409 from 100 trees`.

- The **points score** gives the complete reason, line by line. Nothing is hidden.
- **Logistic regression** gives exact contributions, as in [Explain predictions](https://learning.nextia-ai.com/courses/ml/m07/explain-predictions/#one-tickets-contributions): priority 1 adds +1.23 to the sum, the payment team +0.38, the phone channel +0.23, and the home segment takes 0.71 away.
- **Boosting** gives a score from 100 small trees. scikit-learn has no exact way to split it into reasons. You can ask "what if?" questions, one at a time, as the next cell does.

In [ ]:
changed = ticket.assign(priority=3)
print("points:", int(points(changed).iloc[0]))
print("logistic:", round(logistic.predict_proba(changed[FEATURES])[0, 1], 3))
print("boosting:", round(boosting.predict_proba(changed[FEATURES])[0, 1], 3))

> **Check.** You should see `points: 2`, `logistic: 0.092` and `boosting: 0.095`.

All three agree that priority matters most for this ticket. But only the points score and logistic regression give a full list of reasons that adds up to the score. For boosting, each "what if" is one more question, and the answers do not add up to the score.

## 5. What the complexity costs

The next cell saves each model into memory with joblib, the same way that [Package artifacts](https://learning.nextia-ai.com/courses/ml/m08/package-artifacts/) saves the final model, and prints its size.

In [ ]:
import io
import joblib


def size_kb(model):
    """The size of a saved model, in kilobytes."""
    buffer = io.BytesIO()
    joblib.dump(model, buffer)
    return round(buffer.tell() / 1024)


print("logistic:", size_kb(logistic), "KB,", logistic.named_steps["model"].coef_.size, "coefficients")
print("boosting:", size_kb(boosting), "KB,", boosting.named_steps["model"].n_iter_, "trees")

> **Check.** You should see about 5 KB and 28 coefficients for logistic regression, and about 74 KB and 100 trees for boosting. The sizes can differ a little with other package versions.

Twenty-eight numbers fit on one page, and a reviewer can read each one. A hundred trees with 400 leaves do not. The size is small in both cases; the difference is what a person can check.

## 6. Change one thing: less data

Larkfield has 16,939 train tickets. A smaller shop might have 500. With less data, a flexible model has fewer examples to learn its many splits, and it fits more of the noise.

> **Predict.** Train both models on a random sample of 500 train tickets. Which one loses more average precision?

> **Your turn.** Make `small = train.sample(500, random_state=0)`. Fit a new `build_pipeline()` (logistic) and a new boosting pipeline (same settings as above) on `small`. Put the valid AP of each, rounded to 3 decimals, in `ap_small_logistic` and `ap_small_boosting`. Run the cell, then the check cell.

In [ ]:
small = train.sample(500, random_state=0)
small_logistic = build_pipeline().fit(small[FEATURES], small[TARGET])
small_boosting = build_pipeline(HistGradientBoostingClassifier(max_depth=2, learning_rate=0.1, random_state=0))
small_boosting.fit(small[FEATURES], small[TARGET])
ap_small_logistic = round(average_precision_score(valid[TARGET], small_logistic.predict_proba(valid[FEATURES])[:, 1]), 3)
ap_small_boosting = round(average_precision_score(valid[TARGET], small_boosting.predict_proba(valid[FEATURES])[:, 1]), 3)
print(ap_small_logistic, ap_small_boosting)

In [ ]:
expect_hash('the two AP values on 500 tickets', None if ... in (ap_small_logistic, ap_small_boosting) else [ap_small_logistic, ap_small_boosting], '428136d9425c3d14')

Compare with the full train set (0.391 and 0.395). The [chart on the lesson page](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/#small-data) shows the same comparison for seven sizes, averaged over five samples.

## 7. Guided practice: apply the one-standard-error rule

In [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/#the-one-standard-error-rule), the best setting (boosting) had a cross-validation AP of 0.3424, and logistic regression with C = 1 had 0.3393. The standard error of the best score was 0.0080.

> **Your turn.** Put the difference between the two scores, rounded to 4 decimals, in `gap`. Then set `keep_simple` to `True` if the gap is smaller than one standard error, and `False` if not.

In [ ]:
gap = round(0.3424 - 0.3393, 4)
keep_simple = gap < 0.0080
print(gap, keep_simple)

In [ ]:
expect_hash('the gap and the decision', None if ... in (gap, keep_simple) else [gap, keep_simple], 'bfbad7cf882d3173')

Now write Priya's model-choice note in the next cell (double-click to edit): three to five sentences that say which model Larkfield keeps, the evidence, and what a complex model would cost. Use the checklist on the [lesson page](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/#a-decision-checklist). The lesson page has an example answer.

*Your note here:*

## 8. Failure case: the same reasons for everyone

Omar wants each customer whose ticket is flagged to get a short note with the reasons. Tomás suggests: "Keep boosting, and use its three most important features as the reasons." The next cell does that for two very different tickets: T-117014 (priority 1, payment, flagged) and T-117033 (priority 3, returns, not flagged).

> **Predict.** Will the two tickets get different reasons?

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(boosting, valid[FEATURES], valid[TARGET], scoring="average_precision",
                                n_repeats=5, random_state=0)
top3 = pd.Series(result.importances_mean, index=FEATURES).nlargest(3).index.tolist()
for ticket_id in ["T-117014", "T-117033"]:
    row = valid[valid["ticket_id"] == ticket_id]
    score = boosting.predict_proba(row[FEATURES])[0, 1]
    print(ticket_id, "score", round(score, 3), "| reasons:", ", ".join(top3))

> **Check.** You should see the same three reasons, `priority, team, word_count`, for both tickets, with scores of 0.409 and 0.032.

Permutation importance describes the whole model, not one ticket. It says "priority matters", not whether this ticket's priority pushed the score up or down. So one ticket hears "priority" as the reason it was flagged, and another hears "priority" as the reason it was not. Reasons for one decision must come from that decision: the points of that ticket, or the contributions of that ticket in a linear model. The [lesson page](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/#failure-case-the-same-reasons-for-everyone) has the full case.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson you built a three-line points score from the train rates and compared it with logistic regression and boosting at the same 20% capacity. Logistic regression and boosting both catch 86 of 183 escalations; the points score catches 53 at its cut-off, or 84 with a tie rule. Only the points score and logistic regression give exact reasons for one ticket. With 500 train tickets, boosting falls further behind. Next, you leave the target behind and look for groups of customers.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Unsupervised exploration](https://learning.nextia-ai.com/courses/ml/m07/unsupervised-exploration/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m07/when-a-simple-model-is-better/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.